setup and the Auto Loader function

The same folders and ingest function as in the last notebook (a new notebook needs its own copy).

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"
base = "/Volumes/workspace/de_ecommerce/raw_files"
holding = f"{base}/holding"
incoming = f"{base}/incoming"
ckpt = f"{base}/_checkpoints"

tables = ["orders", "order_items", "order_payments", "order_reviews"]

def count(t):
    return spark.table(f"{catalog}.{schema}.bronze_{t}").count()

def ingest(t):
    target = f"{catalog}.{schema}.bronze_{t}"
    cols = spark.table(target).columns
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{ckpt}/{t}/schema")
        .load(f"{incoming}/{t}/")
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingested_at", F.current_timestamp())
        .drop("_rescued_data")
        .select(*cols)
        .writeStream
        .option("checkpointLocation", f"{ckpt}/{t}/checkpoint")
        .trigger(availableNow=True)
        .toTable(target)
        .awaitTermination())

counts before the batch arrives

Records how many rows each bronze table has right now, so we can compare afterwards.

In [0]:
before = {t: count(t) for t in tables}
print(before)

{'orders': 45430, 'order_items': 51234, 'order_payments': 47871, 'order_reviews': 45370}


deliver batch_1

Copies the batch_1 folder of each table from holding into incoming, as if the source system had just sent a new file.

In [0]:
batch = "batch_1"

for t in tables:
    dbutils.fs.cp(f"{holding}/{t}/{batch}", f"{incoming}/{t}/{batch}", recurse=True)

display(dbutils.fs.ls(f"{incoming}/orders"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/batch_1/,batch_1/,0,1793136246803
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/,initial/,0,1793136246803


run Auto Loader and check

Runs ingest for each table. Then, for each table, it compares the new row count with the old count plus the size of the batch. A line saying OK means exactly the new rows were added, with nothing missing and nothing doubled.

In [0]:
for t in tables:
    ingest(t)

for t in tables:
    new_rows = spark.read.parquet(f"{holding}/{t}/{batch}").count()
    after = count(t)
    status = "OK" if after == before[t] + new_rows else "MISMATCH"
    print(f"{t}: {before[t]} + {new_rows} = {after}  [{status}]")

orders: 45430 + 21208 = 66638  [OK]
order_items: 51234 + 24097 = 75331  [OK]
order_payments: 47871 + 22027 = 69898  [OK]
order_reviews: 45370 + 21190 = 66560  [OK]


see which files the rows came from

Shows how many bronze_orders rows came from each file. This is the proof that both the initial chunk and the new batch are in the table.

In [0]:
display(spark.table(f"{catalog}.{schema}.bronze_orders")
        .groupBy("source_file").count())

source_file,count
/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/part-00000-tid-7866998030968877259-cbfc3878-f618-4262-97da-92ec48e105d0-189-1.c000.snappy.parquet,45430
/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/batch_1/part-00000-tid-8850710974723624380-774578a0-09d0-452d-a898-b30b7124afaa-191-1.c000.snappy.parquet,21208
